# 80장 · Codex 요청의 수용 기준과 JavaScript 상태 변경

같은 제목인 복습을 두 번 등록하면 버튼은 어느 항목을 가리켜야 할까요?

Codex로 투두 앱을 구현할 때 사용할 수용 기준을 작성합니다. 같은 제목을 두 번 입력한 경우와 공백만 입력한 경우로 항목 ID, 초기 상태, 추가 거부를 검증합니다.

실행을 확인한 커널은 Python 3.12.15입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 할 일 추가 함수와 빈 입력 검증

Codex 요청 예시입니다. ‘단일 사용자용 투두 화면에서 제목 양끝의 공백을 제거하고 새 항목을 추가해 줘. 빈 제목은 거절하고 ID는 저장소가 발급해. 같은 제목 두 건을 각각 완료할 수 있어야 해. 변경할 함수와 확인할 사례를 먼저 설명해 줘.’ 아래 코드는 이 요청의 수용 조건을 검토할 수 있도록 마련한 수업용 예시입니다. Codex가 실제로 돌려준 코드는 직접 실행한 세션에서 확인하세요.


### 요구한 동작을 출력으로 확인하기

<p>추가 함수가 완성되었다는 기준을 값으로 정해 보세요. 앞뒤에 공백이 있는 복습을 넣으면 제목은 복습, ID는 1, 완료는 false여야 합니다. 공백만 넣었을 때는 배열 길이가 증가하지 않아야 합니다.</p>


### ID로 같은 제목 구분하기

<p>서로 다른 항목에 복습이라는 제목을 두 번 넣어도 ID는 1과 2로 나뉩니다. 완료 변경의 대상을 정할 때는 제목을 비교하기보다 ID로 해당 행을 찾으세요.</p>


### 셀 01 · 할 일 추가와 반환 레코드

수업 커널의 Node.js로 실제 JavaScript를 실행합니다. 새 레코드의 시작 상태를 보세요.


In [1]:
import subprocess
def run_js(source):
    run = subprocess.run(["node", "--input-type=module", "-e", source],
                         text=True, capture_output=True, check=True, timeout=10)
    print(run.stdout.rstrip())
store_source = """
let nextId = 1;
const items = [];
function add(raw) {
  if (typeof raw !== "string" || raw.trim() === "") throw Error("INVALID_TITLE");
  const item = {id: nextId++, title: raw.trim(), completed: false};
  items.push(item);
  return item;
}
"""
run_js(store_source + 'console.log(JSON.stringify(add("  복습  ")));')

{"id":1,"title":"복습","completed":false}


입력한 ‘  복습  ’은 제목 ‘복습’으로 저장됩니다. trim은 양끝 공백만 없애므로 ‘강의 복습’을 넣으면 두 단어 사이의 공백이 남습니다. 같은 제목을 두 번 추가해도 새 ID가 발급됩니다.


### run_js를 다시 호출할 때의 배열

<p>run_js는 호출마다 별도의 Node 프로세스를 실행합니다. 같은 store_source를 다시 전달하면 nextId는 1, items는 빈 배열에서 다시 시작합니다. 서로 다른 셀의 목록을 한 목록처럼 이어서 읽지 마세요.</p>


## 버전 검사로 투두 변경 한 번만 승인하기

79장에서 고른 ID만 완료 처리합니다. 두 탭에서 같은 버전을 보고 변경을 요청했을 때 첫 승인 뒤에 도착한 요청이 기존 결과를 덮는지 확인합니다.

노트북을 같은 수업 폴더에 모아 열거나 압축을 푼 장별 폴더에서 실행하세요. 시작 셀은 service-lab/selection.json을 먼저 엽니다. 파일이 없을 때만 셀에 적힌 복귀 예시를 읽습니다.

In [ ]:
from pathlib import Path
import json
here = Path.cwd()
# 각 장 폴더에서 열거나 노트북을 한 폴더에 모아 두어도 같은 실습 폴더를 씁니다.
course = here.parent if here.name in ("daily",) or (here.name.startswith("chapter") and here.name[7:].isdigit()) else here
lab = course / "service-lab"
lab.mkdir(exist_ok=True)
incoming = course / "service-lab/selection.json"
if incoming.exists():
    previous = json.loads(incoming.read_text(encoding="utf-8"))
    print("내 실습 파일", incoming.name)
else:
    previous = {'items': [{'id': 1,
                'title': '변경 검토',
                'candidate': '42ec606abc92ce832234dded6ceb188e2a0f771365698f2b6d2215d7be9aebb2'},
               {'id': 2, 'title': '변경 검토', 'candidate': 'revision-b'}],
     'proposal': {'id': 2, 'candidate': 'revision-b', 'done': True}}
    print("복귀 예시", incoming.name)

내 실습 파일 selection.json


## 버전 검사로 투두 변경 한 번만 승인하기 실습

앞에서 확인한 최소 예제에 내 실습 파일의 값을 넣습니다.

In [ ]:
items = {row["id"]: {**row, "done": False, "version": 0} for row in previous["items"]}
proposal = previous["proposal"]
approval_events = [("read", proposal["id"])]
applied_ids = []
def confirm(item_id, expected_version, done):
    row = items[item_id]
    if row["version"] != expected_version:
        return 409
    approval_events.append(("approve", item_id))
    row.update(done=done, version=expected_version + 1)
    applied_ids.append(item_id)
    approval_events.append(("write", item_id))
    return 200

first = confirm(proposal["id"], 0, proposal["done"])
again = confirm(proposal["id"], 0, proposal["done"])
result = {"item_id": proposal["id"], "statuses": [first, again], "version": items[proposal["id"]]["version"],
          "done": items[proposal["id"]]["done"], "mode": "local-version-test", "events": approval_events, "writes": applied_ids}
print(first, again, result["version"])

200 409 1


첫 호출은 버전을 1로 올립니다. 두 번째 호출이 들고 온 버전은 여전히 0이므로 409에서 멈춥니다. 아래 실제 Todo 프로젝트에서는 같은 요청을 API로 보내 DB에 저장된 버전까지 비교하세요.

### 직접 해 보기

선택하지 않은 항목의 완료 상태와 버전을 출력하세요. 둘 다 초기값이어야 합니다.

In [ ]:
# proposal의 ID를 제외한 항목을 읽으세요.

<details class="notebook-answer"><summary>정답 코드와 실행 결과</summary><pre><code>other = next(row for key, row in items.items() if key != proposal[&quot;id&quot;])
print(other[&quot;done&quot;], other[&quot;version&quot;])</code></pre><pre>False 0</pre><p>버전 갱신은 선택한 항목에서만 일어납니다.</p></details>

### 81장에서 다시 열 파일

방금 계산한 값을 service-lab/approval.json에 저장합니다. 같은 장에서 조건을 바꾸고 다시 저장하면 다음 장도 그 값을 읽습니다.

In [ ]:
destination = lab / "approval.json"
destination.parent.mkdir(parents=True, exist_ok=True)
destination.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(destination.name)

approval.json


### 셀 02 · 빈 문자열 입력 검증

공백만 보낸 경우 저장소 길이도 함께 확인합니다.

run_js는 부를 때마다 새 Node 프로세스에서 시작합니다. 이 셀의 배열은 셀 01에서 추가한 항목을 기억하지 않습니다. 같은 행에 연속으로 한 조작을 확인하려면 그 조작을 한 번의 run_js 안에 넣으세요.


In [2]:
run_js(store_source + '''
try { add("   "); }
catch (error) { console.log(error.message + " " + items.length); }
''')

INVALID_TITLE 0


INVALID_TITLE 0입니다. 저장 전에 검사해서 빈 항목과 불필요한 ID가 생기지 않았습니다.


#### 빈 제목이 들어온 경우의 수용 기준

<mark>오류 메시지뿐 아니라 목록 길이가 그대로인지 검사합니다.</mark> 공백만 있는 제목을 보내고 INVALID_TITLE과 항목 수를 함께 읽으세요. 오류를 표시하면서 목록에는 빈 할 일이 추가되는 결함까지 찾을 수 있습니다.

선수 개념 · 79장 JavaScript 폼 이벤트와 fetch 응답 처리

### 검증이 ID 발급보다 먼저인 이유

<p>add는 입력이 문자열이고 trim 결과가 비어 있지 않은지 먼저 확인합니다. 조건을 어기면 nextId++와 items.push에 도달하기 전에 예외를 던지므로 빈 입력은 목록에 나타나지 않습니다.</p>


## diff 검토와 ID별 완료 처리

제안에 items.find(row => row.title === title)가 들어갔다고 가정하세요. 제목이 같은 두 항목 중 첫 번째만 찾으므로 사용자가 누른 두 번째 행도 구별할 ID를 받아야 합니다.


### 과제 03 · ID별 완료 상태 변경

같은 제목의 할 일을 두 건 추가하세요. 함수가 돌려준 두 번째 항목의 ID로 대상을 찾아 완료로 바꾸고 두 행의 ID와 완료 상태를 출력하세요.


In [3]:
# run_js 한 번 안에서 항목 두 건을 추가하고 두 번째 항목만 변경하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>run_js(store_source + &quot;&quot;&quot;
const first = add(&quot;복습&quot;);
const second = add(&quot;복습&quot;);
const selected = items.find(row =&gt; row.id === second.id);
selected.completed = true;
console.log(JSON.stringify(items.map(row =&gt; [row.id, row.completed])));
&quot;&quot;&quot;)</code></pre>
<p>실행 결과</p><pre><code>[[1,false],[2,true]]</code></pre>
<p>제목으로 첫 일치 항목을 고르면 ID 1을 잘못 바꿉니다. 방금 추가한 항목이 돌려준 ID를 사용하면 nextId의 시작값을 바꿔도 두 번째 행만 완료됩니다.</p>
</details>


### 완료 수정의 범위 검토하기

<p>두 번째 항목을 완료로 바꾼 뒤 두 행의 ID와 완료 값을 함께 출력해 보세요. 목표는 ID 2의 완료 값 하나를 바꾸는 것입니다. 제목이 같은 ID 1까지 바뀌었다면 대상을 찾는 조건부터 살펴보세요.</p>


## 동일한 제목의 할 일을 구분해 삭제하기

Codex에 “삭제 버튼을 만들어 줘”라고만 하면 제목으로 첫 항목을 찾아 지우는 코드도 요구를 만족한 것처럼 보입니다. 같은 제목 두 건을 넣어 사용자가 선택한 항목만 지워지는지 직접 검사합니다.

### 셀 05 · 항목 두 개가 같은 제목을 가진 경우 재현하기

셀 01을 실행하면 run_js와 초기 저장 코드 store_source가 준비됩니다. 두 번째 복습을 선택했는데 제목으로 첫 일치 항목을 삭제하는 구현을 실행해 보세요.

In [4]:
run_js(store_source + """
const first = add("복습");
const selected = add("복습");
const position = items.findIndex(item => item.title === selected.title);
items.splice(position, 1);
console.log(["선택", selected.id, "남은 ID", items.map(item => item.id).join(",")].join(" "));
""")

선택 2 남은 ID 2


2번을 선택했는데 남은 ID가 2입니다. 실제로 지운 항목은 제목이 먼저 일치한 1번입니다. 목록에 제목 한 종류만 출력하면 화면만 보면서 이 오류를 발견하기 어렵습니다.

### 과제 06 · Codex에게 전달할 요구를 ID 기반 삭제로 구현하기

“행의 삭제 버튼은 그 행의 ID 한 건만 제거한다. 없는 ID는 목록을 유지한다.” 이 요구를 removeById 함수로 작성하세요. 두 번째 복습을 두 번 삭제하고 반환값과 남은 ID를 출력하세요.

In [5]:
# store_source 뒤에 removeById(id)를 작성하세요.
# 같은 제목의 두 번째 항목 ID로 두 차례 호출하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>run_js(store_source + &quot;&quot;&quot;
function removeById(id) {
  const at = items.findIndex(item =&gt; item.id === id);
  if (at &lt; 0) return false;
  items.splice(at, 1);
  return true;
}
add(&quot;복습&quot;);
const second = add(&quot;복습&quot;);
console.log([removeById(second.id), removeById(second.id)].join(&quot; &quot;));
console.log(items.map(item =&gt; item.id).join(&quot;,&quot;));
&quot;&quot;&quot;)</code></pre>
<p>실행 결과</p><pre><code>true false
1</code></pre>
<p>첫 줄은 true false이고 남은 번호는 1입니다. 두 번째 삭제에서는 findIndex가 -1을 돌려줍니다. 이때 그대로 splice(-1, 1)을 부르면 마지막 항목까지 지워지므로 먼저 반환해야 합니다.</p>
</details>

<mark>사용자가 선택한 행은 제목이 같아도 ID로 구별됩니다.</mark> 코드 검토에서는 제목이 같을 때와 이미 삭제한 ID를 다시 받을 때를 모두 확인하세요. 두 번째 삭제가 남은 행을 보존한다는 조건까지 있어야 합니다.

81장의 DELETE 요청 주소에 제목 대신 선택한 행의 ID를 넣고, 서버 응답을 받은 뒤 화면 목록을 갱신합니다.

## 돌아보기

코드 검토에선 제목이 같은 두 행을 남겨 두세요. 화면에서 두 번째 행을 눌렀는데 첫 번째가 완료된다면 변경 함수의 검색 조건과 화면이 넘긴 ID를 맞춰 봅니다.
